## Librerias

In [34]:
# Instalar requests si no lo tenemos -> !pip install requests

import json
import requests
import re

## .json

In [28]:
with open("data.json", "r", encoding="utf-8") as f:
    data = json.load(f)


## Modelo

In [29]:
def json_a_texto(data):
    textos = []
    
    for item in data:
        if isinstance(item, dict):
            texto = " ".join([f"{k}: {v}" for k, v in item.items()])
        else:
            texto = str(item)
            
        textos.append(texto)
    
    return textos

corpus = json_a_texto(data)

In [23]:
def buscar_contexto(pregunta, corpus, top_k=3):
    resultados = []
    
    for texto in corpus:
        score = sum(palabra in texto.lower() for palabra in pregunta.lower().split())
        resultados.append((score, texto))
    
    resultados.sort(reverse=True)
    
    return [texto for _, texto in resultados[:top_k]]

## Llama3

In [35]:
def preguntar_llama(pregunta, contexto):
    url = "http://localhost:11434/api/generate"
    
    prompt = f"""
Responde SOLO con la información del contexto.

Contexto:
{contexto}

Pregunta:
{pregunta}
"""

    payload = {
        "model": "llama3",
        "prompt": prompt,
        "stream": False
    }

    response = requests.post(url, json=payload)
    
    if response.status_code == 200:
        return response.json()["response"]
    else:
        return f"Error: {response.status_code}"

In [25]:
def chat(pregunta):
    contextos = buscar_contexto(pregunta, corpus)
    contexto_unido = "\n".join(contextos)
    
    respuesta = preguntar_llama(pregunta, contexto_unido)
    
    return respuesta

In [ ]:
while True:
    pregunta = input("Vos: ")
    
    if pregunta.lower() == "salir":
        break
    
    respuesta = chat(pregunta)
    print("Bot:", respuesta)

    # 'salir' para terminar el chat

Bot: Hola!
Bot: La inteligencia artificial (IA) se refiere a cualquier tecnología que pueda realizar tareas que tradicionalmente requieren la inteligencia humana, como aprender, razonar y tomar decisiones.
Bot: No hay información para responder.
